# 05 · Yield-gap model: Closing Rwanda's yield gap
**NISR Big Data Hackathon 2026 · Track 1 (Agricultural Productivity)**

Plot-level model of crop yield from NISR Seasonal Agricultural Survey microdata (2022–2025, Seasons A and B), with district soil/terrain and seasonal satellite anomalies. Steps:

1. Load the model dataset (from `build_model_dataset.py`)
2. Explore yields and input adoption
3. Validate: unseen segments, unseen districts, unseen year (2025)
4. Train the final model
5. Drivers of yield (SHAP with LightGBM, otherwise permutation importance)
6. Scenarios: yield gain and extra production if non-adopters adopt fertilizer, improved seed, manure, lime, anti-erosion works
7. Fertilizer dose-response
8. Save outputs for the dashboard

All functions live in `src/train_model.py`, so the notebook and the script always give the same results.

In [ ]:
import json, pickle, time
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, importlib
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")          # run from the repository root
sys.path.insert(0, "src")
import train_model as tm
tm = importlib.reload(tm)   # always use the latest train_model.py, even without a kernel restart

DATA = "data/model_data/model_dataset.csv"
OUT = Path("data/model_outputs"); OUT.mkdir(exist_ok=True)
QUICK = False          # True = fast test (fewer trees, sampled training in validation)
MONOTONE = True        # inputs can only raise yield in the model
EXCLUDE = ["Paddy rice"]
opts = SimpleNamespace(quick=QUICK, no_monotone=not MONOTONE)

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
import sys
print("Python:", sys.executable)
print("Engine:", "LightGBM (exact SHAP)" if tm.HAVE_LGB else f"scikit-learn -- LightGBM not loaded: {tm.LGB_ERROR}")

## 1 · Load the model dataset

In [ ]:
d, feats = tm.load(DATA, EXCLUDE)
cats = {c: d[c].unique() for c in tm.CAT if c in feats}
print(f"{len(d):,} plots | {len(feats)} features | years {d.year.min()}-{d.year.max()}")
groups_of = {g: [f for f in v if f in feats] for g, v in tm.GROUPS.items()}
for g, v in groups_of.items():
    print(f"  {g:34s} {len(v):2d}: {', '.join(v)}")
d.pivot_table(index=["year", "season"], columns="crop", values="yield_kg_ha", aggfunc="size", fill_value=0)

## 2 · Explore: yields and input adoption

In [ ]:
w = lambda g, c: np.average(g[c].dropna(), weights=g.loc[g[c].notna(), "w_area_ha"]) if g[c].notna().any() else np.nan
summary = d.groupby("crop").apply(lambda g: pd.Series({
    "plots": len(g), "yield_kg_ha": np.average(g.yield_kg_ha, weights=g.w_area_ha),
    "inorganic_fert_%": 100 * w(g, "inorganic_fert"), "improved_seed_%": 100 * w(g, "improved_seed"),
    "manure_%": 100 * w(g, "organic_fert"), "lime_%": 100 * w(g, "lime"),
    "fert_dose_users_kg_ha": g.loc[g.inorganic_kg_ha > 0, "inorganic_kg_ha"].median()})).round(1)
summary.sort_values("plots", ascending=False)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
order = d.groupby("crop").yield_kg_ha.median().sort_values().index
ax[0].boxplot([d.loc[d.crop == c, "yield_kg_ha"] for c in order], orientation="horizontal", showfliers=False)
ax[0].set_yticks(range(1, len(order) + 1), order); ax[0].set_xscale("log")
ax[0].set_xlabel("Yield (kg/ha, log scale)"); ax[0].set_title("Plot yields by crop (2022-2025)")
m = d[d.crop == "Maize"].assign(f=lambda x: x.inorganic_fert.map({1: "fertilizer", 0: "no fertilizer"}))
for lab, g in m.groupby("f"):
    ax[1].hist(np.log10(g.yield_kg_ha), bins=50, alpha=.55, density=True, label=lab)
ax[1].set_xlabel("log10 maize yield (kg/ha)"); ax[1].set_title("Maize: plots with vs without fertilizer"); ax[1].legend()
plt.tight_layout(); plt.show()

## 3 · Validation
Three tests, each against a **baseline** = average yield of that crop in that season:
- **random**: whole survey segments held out (5 folds)
- **district**: whole districts held out (6 folds of 5 districts), the key test for mapping yield gaps in districts the model has not seen
- **temporal**: trained on 2022–2024, tested on 2025

`plot_R2_within_crop` = share of the plot-to-plot variation **within** a crop and season that the model explains. District scores use survey-weighted district mean yields (district × season × crop cells with at least 20 plots).

In [ ]:
t0 = time.time(); vlog = []
metrics, oof = tm.evaluate(d, feats, cats, opts, vlog)
print("\n".join(vlog), f"\ntotal {time.time()-t0:.0f}s")
metrics.to_csv(OUT / "validation_metrics.csv", index=False); oof.to_csv(OUT / "oof_predictions.csv", index=False)
print("district_R2_within_crop = can the model rank districts for the SAME crop and season? (baseline scores 0)")
metrics.set_index("validation").T.round(3)

In [ ]:
obs, prd = tm.district_level(oof, "p_model")
cell = pd.DataFrame({"observed": obs, "predicted": prd}).reset_index()
crops_show = cell.crop.value_counts().index[:6]
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, c in zip(axes.ravel(), crops_show):
    g = cell[cell.crop == c]
    ax.scatter(g.observed, g.predicted, s=12, alpha=.6)
    lim = [min(g.observed.min(), g.predicted.min()), max(g.observed.max(), g.predicted.max())]
    ax.plot(lim, lim, "k--", lw=.8)
    ax.set_title(f"{c}  (r = {np.corrcoef(g.observed, g.predicted)[0,1]:.2f}, n = {len(g)})")
    ax.set_xlabel("observed district yield (kg/ha)"); ax.set_ylabel("predicted")
fig.suptitle("District hold-out: predicted vs observed district yields (districts never seen in training)")
plt.tight_layout(); plt.show()

## 4 · Train the final model on all data

In [ ]:
model = tm.Model(feats, cats, MONOTONE, QUICK).fit(d)
pickle.dump(model, open(OUT / "yield_model.pkl", "wb"))
json.dump({"features": feats, "categories": {k: list(v) for k, v in model.cats.items()}, "engine": model.engine,
           "smearing": model.smear, "monotone_up": tm.MONOTONE_UP if MONOTONE else []},
          open(OUT / "model_features.json", "w"), indent=1)
print(model.engine, "| smearing factor", round(model.smear, 3))

## 5 · What drives yield differences within a crop

In [ ]:
dlog = []
fi, by_crop, driver_groups = tm.drivers(model, d, feats, opts, dlog)
print(dlog[0])
fi.to_csv(OUT / "feature_importance.csv", index=False); by_crop.to_csv(OUT / "importance_by_crop.csv", index=False)
driver_groups.to_csv(OUT / "driver_groups.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1.6]})
dg = driver_groups.sort_values("share_pct")
ax[0].barh(dg.group, dg.share_pct); ax[0].set_xlabel("% of importance (crop identity excluded)")
ax[0].set_title("Driver groups")
top = fi[fi.share_pct.notna()].head(20).iloc[::-1]
colors = {g: f"C{i}" for i, g in enumerate(tm.GROUPS)}
ax[1].barh(top.feature, top.share_pct, color=top.group.map(colors))
ax[1].set_xlabel("% of importance"); ax[1].set_title("Top 20 features")
from matplotlib.patches import Patch
ax[1].legend(handles=[Patch(color=c, label=g) for g, c in colors.items() if g != "Crop & season"],
             fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

In [ ]:
# importance per crop (top 12 features overall, crop identity excluded)
keep = fi[fi.share_pct.notna()].feature.head(12)
bc = by_crop.set_index("feature").loc[keep]
(100 * bc / by_crop.set_index("feature").drop(index=[f for f in ["crop", "season"] if f in by_crop.feature.values]).sum()).round(1)

## 6 · Scenarios: closing the yield gap
For the latest survey year, each plot is predicted as it is and again **as if** it used the input (fertilizer at the crop's typical dose, i.e. the median of users). Gains are aggregated with survey weights to district × season × crop. A lever is scored for a crop only if the survey has ≥ 200 plots with and without it.

In [ ]:
slog = []
dist, nat, top_targets = tm.scenarios(model, d, int(d.year.max()), slog)
print("\n".join(slog))
dist.round(3).to_csv(OUT / "scenario_district.csv", index=False); nat.round(3).to_csv(OUT / "scenario_national.csv", index=False)
top_targets.round(3).to_csv(OUT / "top_targets.csv", index=False)
print("\nNational yield gain (%) if non-adopters adopt:")
nat.pivot_table(index=["crop", "season"], columns="scenario", values="yield_gain_pct").round(1)

In [ ]:
print("Extra production (tonnes, Seasons A+B):")
nat.pivot_table(index="crop", columns="scenario", values="extra_production_t", aggfunc="sum").round(0)

In [ ]:
crop, season, scen = "Maize", "A", "package_fert_seed_manure"
g = dist[(dist.crop == crop) & (dist.season == season) & (dist.scenario == scen) & (dist.plots >= 20)]
g = g.sort_values("yield_gain_pct")
fig, ax = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
ax[0].barh(g.district, g.current_yield_kg_ha, label="current"); ax[0].barh(g.district, g.attainable_yield_kg_ha - g.current_yield_kg_ha, left=g.current_yield_kg_ha, label="gap (attainable)")
ax[0].set_xlabel("kg/ha"); ax[0].legend(); ax[0].set_title(f"{crop} Season {season}: current vs attainable yield")
ax[1].barh(g.district, g.extra_production_t, color="C2"); ax[1].set_xlabel("extra tonnes")
ax[1].set_title("Extra production if non-adopters adopt fertilizer + seed + manure")
plt.tight_layout(); plt.show()
top_targets[top_targets.scenario == scen].head(15)[["season", "district", "crop", "levers", "adoption_pct", "yield_gain_pct", "extra_production_t"]].round(1)

## 7 · Fertilizer dose-response (plots that currently use no inorganic fertilizer)

In [ ]:
dr = tm.dose_response(model, d, int(d.year.max()))
dr.round(2).to_csv(OUT / "dose_response.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 5))
for c, g in dr.groupby("crop"):
    ax.plot(g.kg_per_ha, g.yield_gain_pct, marker="o", ms=3, label=c)
ax.set_xlabel("inorganic fertilizer (kg/ha)"); ax.set_ylabel("expected yield gain (%)"); ax.legend(fontsize=8)
ax.set_title("Model dose-response by crop"); plt.show()
dr.pivot(index="kg_per_ha", columns="crop", values="yield_gain_pct").round(1)

## 8 · Notes for the documentation
- Effects are **model-based associations from survey data**, not field-trial results. Positive effects are imposed (monotone constraints) for improved seed, manure, inorganic fertilizer and dose, lime and anti-erosion works.
- Results should be read at **district level**: plot-level yields are noisy (see `plot_R2_within_crop`).
- Satellite and soil features are district-level because NISR plot locations are not public; with GPS-linked plots, the same pipeline could add plot-level terrain (TWI, curvature) and NDWI.
- 2024 Season B uses only plots with an exact crop share (NISR screening file lists one crop per plot).
- All files in `model_outputs/` are aggregated and can be published; raw microdata stays off GitHub.

In [ ]:
print("Saved:", ", ".join(sorted(p.name for p in OUT.iterdir())))